In [7]:
import gym
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from collections import deque
import random

# 1. Define the DQN Agent
class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000)  # Experience replay memory
        self.gamma = 0.95                 # Discount factor
        self.epsilon = 1.0                # Exploration rate
        self.epsilon_min = 0.01           # Minimum exploration rate
        self.epsilon_decay = 0.995        # Exploration decay rate
        self.learning_rate = 0.001        # Learning rate
        self.model = self._build_model()

    # 2. Build the neural network model
    def _build_model(self):
        model = Sequential()
        model.add(Dense(24, input_dim=self.state_size, activation='relu'))  # Hidden layer 1
        model.add(Dense(24, activation='relu'))                             # Hidden layer 2
        model.add(Dense(self.action_size, activation='linear'))             # Output layer
        model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate=self.learning_rate))
        return model

    # 3. Store the experience in memory
    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    # 4. Choose action (Explore or Exploit)
    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)  # Explore: random action
        state = np.reshape(state, [1, self.state_size])  # Ensure the state is 2D (1, state_size)
        q_values = self.model.predict(state)  # Exploit: choose action with highest Q-value
        return np.argmax(q_values[0])


    # 5. Learn from experience
    def replay(self, batch_size):
        minibatch = random.sample(self.memory, batch_size)
        for state, action, reward, next_state, done in minibatch:
            target = reward
            if not done:
                target = reward + self.gamma * np.amax(self.model.predict(next_state)[0])
            target_f = self.model.predict(state)
            target_f[0][action] = target
            self.model.fit(state, target_f, epochs=1, verbose=0)
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

# 6. Main function to train the robot
def train_dqn_agent():
    env = gym.make('CartPole-v1')   # Create the environment
    state_size = env.observation_space.shape[0]  #  dimensions (state size)
    action_size = env.action_space.n             # Number of possible actions (left, right)
    agent = DQNAgent(state_size, action_size)
    episodes = 1000  # Number of games/episodes to train

    for e in range(episodes):
        state = env.reset()
        state = np.reshape(state, [1, state_size])  # Reshape for input into the neural network
        for time in range(500):
            env.render()  # Render the environment (comment this if you don't want to see the simulation)
            action = agent.act(state)  # Choose action using the agent's policy
            next_state, reward, done, _ = env.step(action)  # Take action in the environment
            reward = reward if not done else -10  # Penalize if done (episode ends)
            next_state = np.reshape(next_state, [1, state_size])
            agent.remember(state, action, reward, next_state, done)  # Store experience
            state = next_state
            if done:
                print(f"Episode: {e}/{episodes}, Score: {time}, Epsilon: {agent.epsilon}")
                break  # Move to the next episode if the game ends
        if len(agent.memory) > 32:
            agent.replay(32)  # Learn from past experiences

    env.close()

if __name__ == "__main__":
    train_dqn_agent()


ValueError: Invalid dtype: tuple

In [17]:
def train_dqn_agent():
    env = gym.make('CartPole-v1')   # Create the environment
    state_size = env.observation_space.shape[0]  # Input dimensions (state size)
    print(state_size)
    print(env.observation_space)
    action_size = env.action_space.n     
    print(action_size)    # Number of possible actions (left, right)
    agent = DQNAgent(state_size, action_size)
    episodes = 1000 
train_dqn_agent()


4
Box([-4.8000002e+00 -3.4028235e+38 -4.1887903e-01 -3.4028235e+38], [4.8000002e+00 3.4028235e+38 4.1887903e-01 3.4028235e+38], (4,), float32)
2


In [12]:
import gym

# Create an environment
env = gym.make('CartPole-v1')


In [15]:
# Reset the environment to the initial state
state = env.reset()

for _ in range(1000):
    # Render the environment (optional)
     #Sample a random action
    action = env.action_space.sample()

# Take the action and observe the result
    next_state, reward, done, truncated, info = env.step(action)

# Check if the episode is done
    if done:
        break

# Close the environment
env.close()


In [16]:
result = env.step(action)
print(result)


(array([-0.04980292, -0.56999826,  0.2411114 ,  1.4795622 ], dtype=float32), 0.0, True, False, {})


/Users/nipunsingh/Documents/Project/Object Detection/detect/lib/python3.10/site-packages/gym/envs/classic_control/cartpole.py:177: UserWarning: WARN: You are calling 'step()' even though this environment has already returned terminated = True. You should always call 'reset()' once you receive 'terminated = True' -- any further steps are undefined behavior.
  logger.warn(
